## Trending YouTube Dataset Project

## Import Required Libraries

In [36]:
import pandas as pd           # To handle data tables easily
import glob                   # To find specific files in your folders
import os                     # To manage folders and file paths
import json                   # To read those .json data files
from IPython.display import display # To make data look pretty when it prints

import gdown                  # The tool that downloads from Google Drive
import zipfile                # To open up zipped folders

# The specific ID for the file you want from Drive
file_id = "1t9QkQEuvEnZ18VkZsij9fKH7U7x-iu1x"

# The web link created using that ID
url = f"https://drive.google.com/file/d/{file_id}/view?usp=sharing"

# What you want to name the downloaded file
output = "data.zip"

# This part actually starts the download:
# quiet=False: shows you a progress bar so you know it's working
# fuzzy=True: makes sure the link works even if it's a bit messy
gdown.download(url, output, quiet=False, fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1t9QkQEuvEnZ18VkZsij9fKH7U7x-iu1x
From (redirected): https://drive.google.com/uc?id=1t9QkQEuvEnZ18VkZsij9fKH7U7x-iu1x&confirm=t&uuid=c54e1a43-ca0d-436d-bbd7-cc0299d4e604
To: C:\Users\yasee\Downloads\CS_Project.2025-2026\data.zip
100%|██████████| 57.2M/57.2M [00:14<00:00, 3.99MB/s]


'data.zip'

## 1. Load and Concatenate All CSV Files (Add Country Column)

In [52]:
# This is where the folder with all the YouTube data is saved
dataset_path = 'data/trendingYT'

# Look in that folder and make a list of every file that ends with '.zst'
CSVs = [x for x in os.listdir(dataset_path) if x.endswith('.zst')]

# Start an empty table so we have somewhere to put all the data
consolidated_df = pd.DataFrame()

# Now, go through those files one by one
for i in CSVs:
    # Look at the first two letters of the filename to figure out the country
    country = i[:2]
    
    # Open the file and turn it into a readable table
    df = pd.read_csv(
        f"{dataset_path}/{i}",
        compression='zstd',
        encoding='ISO-8859-1'
    )
    
    # Add a new column to the table so we remember which country this data belongs to
    df['country'] = country
    
    # Stack this file's data onto the bottom of our main table
    consolidated_df = pd.concat(
        [consolidated_df, df],
        ignore_index=True,
        axis=0,
        copy=False
    )

# Show the very last few rows of the finished table to check it
display(consolidated_df.tail())

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
375937,BZt0qjTWNhw,18.14.06,The Cat Who Caught the Laser,AaronsAnimals,15,2018-05-18T13:00:04.000Z,"aarons animals|""aarons""|""animals""|""cat""|""cats""...",1685609,38160,1385,2657,https://i.ytimg.com/vi/BZt0qjTWNhw/default.jpg,False,False,False,The Cat Who Caught the Laser - Aaron's Animals,US
375938,1h7KV2sjUWY,18.14.06,True Facts : Ant Mutualism,zefrank1,22,2018-05-18T01:00:06.000Z,[none],1064798,60008,382,3936,https://i.ytimg.com/vi/1h7KV2sjUWY/default.jpg,False,False,False,NaN,US
375939,D6Oy4LfoqsU,18.14.06,I GAVE SAFIYA NYGAARD A PERFECT HAIR MAKEOVER ...,Brad Mondo,24,2018-05-18T17:34:22.000Z,I gave safiya nygaard a perfect hair makeover ...,1066451,48068,1032,3992,https://i.ytimg.com/vi/D6Oy4LfoqsU/default.jpg,False,False,False,I had so much fun transforming Safiyas hair in...,US
375940,oV0zkMe1K8s,18.14.06,How Black Panther Should Have Ended,How It Should Have Ended,1,2018-05-17T17:00:04.000Z,"Black Panther|""HISHE""|""Marvel""|""Infinity War""|...",5660813,192957,2846,13088,https://i.ytimg.com/vi/oV0zkMe1K8s/default.jpg,False,False,False,How Black Panther Should Have EndedWatch More ...,US
375941,ooyjaVdt-jA,18.14.06,Official Call of DutyÂ®: Black Ops 4 âÂ Mult...,Call of Duty,20,2018-05-17T17:09:38.000Z,"call of duty|""cod""|""activision""|""Black Ops 4""",10306119,357079,212976,144795,https://i.ytimg.com/vi/ooyjaVdt-jA/default.jpg,False,False,False,Call of Duty: Black Ops 4 Multiplayer raises t...,US


## 2. Extract All Videos with No Tags

In [53]:
# Look for videos that don't have any tags at all
# This finds two types:
# 1) Videos where the tags literally say "[none]"
# 2) Videos where the tag info is just missing or blank

no_tag_videos = consolidated_df[
    (consolidated_df["tags"] == "[none]") | 
    (consolidated_df["tags"].isna())
]

# Show the first few rows of these tag-less videos
display(no_tag_videos.head())

# Print out exactly how many of these videos we found in total
print(f"Total videos with no tags: {len(no_tag_videos)}")

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
41,JwboxqDylgg,17.14.11,Canada Soccer's Women's National Team v USA In...,Canada Soccer,17,2017-11-13T05:53:49.000Z,[none],36311,277,28,13,https://i.ytimg.com/vi/JwboxqDylgg/default.jpg,False,False,False,Canada Soccer's Women's National Team face riv...,CA
58,9B-q8h31Bpk,17.14.11,John Oliver Tackles Louis C.K. And Donald Trum...,TV Shows,22,2017-11-13T04:49:26.000Z,[none],106029,1270,101,181,https://i.ytimg.com/vi/9B-q8h31Bpk/default.jpg,False,False,False,"John Oliver on News, Politics ...",CA
78,1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subsc...,CA
86,pmJQ4KwliX4,17.14.11,"LATEST Q POSTS: ROTHSCHILDS, HOUSE OF SAUD, lL...",James Munder,2,2017-11-12T21:25:40.000Z,[none],116820,1503,139,1066,https://i.ytimg.com/vi/pmJQ4KwliX4/default.jpg,False,False,False,https://pastebin.ca/3930472\n\nSupport My Chan...,CA
98,lHcXhBojpeQ,17.14.11,ä¸å±TVBè¦å¸ï¼ææ£10å¹´éæ¢ ç«¹é¦¬é«®å¦...,ææç¾æç,22,2017-11-12T12:49:50.000Z,[none],88061,47,58,17,https://i.ytimg.com/vi/lHcXhBojpeQ/default.jpg,False,False,False,NaN,CA


Total videos with no tags: 37698


## 3. Total Number of Views for Each Channel

In [56]:
# Group all the data together based on the channel name
# Then, add up all the views for every video each channel has
views_per_channel = (
    consolidated_df
    .groupby("channel_title")["views"]
    .sum()
    .reset_index()
)

# Show the first few rows so we can see the totals for each channel
display(views_per_channel.head())

,channel_title,views
0,! ì¸ìì ë¬´ì¨ì¼ì´,3942977
1,!!8æã ãé¢ç½ãã¿å¤§éå,50207
2,!BTSã»TWICE ã¾ã¨ã,7310
3,!Los amorosos ViralesÂ¡,6069
4,!t Live,240038


## 4. Create Excluded DataFrame and Remove from Original

In [57]:
# First, let's find all the videos where the comments have been turned off
disabled_comments = consolidated_df[consolidated_df["comments_disabled"] == True]

# Show the first few rows of these videos to see what they look like
display(disabled_comments.head())

# Now, let's pick out all the videos we want to get rid of:
# 1) Ones with comments turned off
# 2) Ones where you can't see the likes or dislikes (ratings)
# 3) Ones that have an error or were deleted from YouTube
excluded = consolidated_df[
    (consolidated_df["comments_disabled"] == True) |
    (consolidated_df["ratings_disabled"] == True) |
    (consolidated_df["video_error_or_removed"] == True)
]

# Show a quick preview of these "bad" or restricted videos
display(excluded.head())

# Actually delete these rows from our main list of data
consolidated_df = consolidated_df.drop(excluded.index)

# Show the first few rows of the final, "clean" list
display(consolidated_df.head())

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
70,TIvI07xLPnA,17.14.11,"The National for Sunday, November 12, 2017",The National,25,2017-11-13T03:06:10.000Z,"Canada|""CBC""|""CBC News""|""National""|""News""|""The...",13433,74,57,0,https://i.ytimg.com/vi/TIvI07xLPnA/default.jpg,True,False,False,"Welcome to The National, the flagship nightly ...",CA
82,Dgut-rlPVrk,17.14.11,Will Grace Davies make you love her? | Live Sh...,The X Factor UK,24,2017-11-12T19:59:14.000Z,"the x factor|""x factor""|""X factor UK""|""x facto...",261603,4276,2148,0,https://i.ytimg.com/vi/Dgut-rlPVrk/default.jpg,True,False,False,Visit the official site: http://itv.com/xfacto...,CA
173,H8IWLEtFt9A,17.14.11,IntÃ©grale - On n'est pas couchÃ© 11 novembre ...,On n'est pas couchÃ©,24,2017-11-12T01:44:07.000Z,"onpc|""on n'est pas couche""|""laurent ruquier""|""...",122282,0,0,0,https://i.ytimg.com/vi/H8IWLEtFt9A/default.jpg,True,True,False,IntÃ©grale - On n'est pas couchÃ© 11 novembre ...,CA
235,wDEA3rpYHnI,17.15.11,Marie-Louise Arsenault rÃ©plique Ã Denise Bom...,TV Classics,22,2017-11-13T01:26:37.000Z,Marie-Louise Arsenault qui rÃ©plique Ã Denise...,15800,88,0,0,https://i.ytimg.com/vi/wDEA3rpYHnI/default.jpg,True,False,False,Moment favori Ã la tÃ©lÃ© quÃ©bÃ©coise: Marie...,CA
371,S-5IU_xdAIg,17.15.11,WATCH LIVE: Attorney General Sessions testifie...,PBS NewsHour,25,2017-11-14T22:07:58.000Z,[none],46905,228,29,0,https://i.ytimg.com/vi/S-5IU_xdAIg/default.jpg,True,False,False,NaN,CA


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
67,amEZKmJQ4Io,17.14.11,Drako - Watch Me Do It [Official Video],babygranderecords,10,2017-10-23T19:38:36.000Z,"Drako|""Watch Me Do It""|""watch me""|""migos""|""dap...",25887,0,0,6,https://i.ytimg.com/vi/amEZKmJQ4Io/default.jpg,False,True,False,PURCHASE / STREAM WATCH ME DO IT https://fanli...,CA
70,TIvI07xLPnA,17.14.11,"The National for Sunday, November 12, 2017",The National,25,2017-11-13T03:06:10.000Z,"Canada|""CBC""|""CBC News""|""National""|""News""|""The...",13433,74,57,0,https://i.ytimg.com/vi/TIvI07xLPnA/default.jpg,True,False,False,"Welcome to The National, the flagship nightly ...",CA
82,Dgut-rlPVrk,17.14.11,Will Grace Davies make you love her? | Live Sh...,The X Factor UK,24,2017-11-12T19:59:14.000Z,"the x factor|""x factor""|""X factor UK""|""x facto...",261603,4276,2148,0,https://i.ytimg.com/vi/Dgut-rlPVrk/default.jpg,True,False,False,Visit the official site: http://itv.com/xfacto...,CA
173,H8IWLEtFt9A,17.14.11,IntÃ©grale - On n'est pas couchÃ© 11 novembre ...,On n'est pas couchÃ©,24,2017-11-12T01:44:07.000Z,"onpc|""on n'est pas couche""|""laurent ruquier""|""...",122282,0,0,0,https://i.ytimg.com/vi/H8IWLEtFt9A/default.jpg,True,True,False,IntÃ©grale - On n'est pas couchÃ© 11 novembre ...,CA
235,wDEA3rpYHnI,17.15.11,Marie-Louise Arsenault rÃ©plique Ã Denise Bom...,TV Classics,22,2017-11-13T01:26:37.000Z,Marie-Louise Arsenault qui rÃ©plique Ã Denise...,15800,88,0,0,https://i.ytimg.com/vi/wDEA3rpYHnI/default.jpg,True,False,False,Moment favori Ã la tÃ©lÃ© quÃ©bÃ©coise: Marie...,CA


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. BeyoncÃ©,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. BeyoncÃ© ...,CA
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...,CA
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO â¶ \n\nSUBSCRIBE âº ...,CA
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...,CA
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,ð§: https://ad.gt/yt-perfect\nð°: https://...,CA


## 5. Add Like Ratio Column (Likes / Dislikes)

In [58]:
# Create a new column to show the ratio of likes to dislikes
# If a video has 0 dislikes, we change that 0 to a blank space (NA) 
# This stops the computer from crashing when it tries to divide by zero
consolidated_df["like_ratio"] = (
    consolidated_df["likes"] / consolidated_df["dislikes"].replace(0, pd.NA)
)

# Show a quick table with just the likes, dislikes, and the new ratio we made
display(consolidated_df[["likes", "dislikes", "like_ratio"]].head())

,likes,dislikes,like_ratio
0,787425,43420,18.135076
1,127794,1688,75.707346
2,146035,5339,27.3525
3,132239,1989,66.485168
4,1634130,21082,77.513044


## 6. Cluster Publish Time into 10-Minute Intervals

In [59]:
# Change the 'publish_time' column into a format the computer understands as a real date and time
# If any dates are broken or look weird, just leave them blank (NaT)
consolidated_df["publish_time"] = pd.to_datetime(
    consolidated_df["publish_time"], errors="coerce"
)

# Create a new column that rounds each upload time down to the nearest 10 minutes
# This makes it easier to group videos that were posted around the same time
consolidated_df["publish_interval"] = (
    consolidated_df["publish_time"].dt.floor("10min")
)

# Show the original time next to our new 10-minute "rounded" time
display(consolidated_df[["publish_time", "publish_interval"]].head())

,publish_time,publish_interval
0,2017-11-10 17:00:03+00:00,2017-11-10 17:00:00+00:00
1,2017-11-13 17:00:00+00:00,2017-11-13 17:00:00+00:00
2,2017-11-12 19:05:24+00:00,2017-11-12 19:00:00+00:00
3,2017-11-12 18:01:41+00:00,2017-11-12 18:00:00+00:00
4,2017-11-09 11:04:14+00:00,2017-11-09 11:00:00+00:00


## 7. Interval Statistics (Video Count, Avg Likes, Avg Dislikes)

In [60]:
# Group the videos together based on those 10-minute time blocks we created
# For each time block, let's calculate:
# - How many videos were posted (video_count)
# - The average number of likes (avg_likes)
# - The average number of dislikes (avg_dislikes)
interval_stats = (
    consolidated_df
    .groupby("publish_interval")
    .agg(
        video_count=("video_id", "count"),
        avg_likes=("likes", "mean"),
        avg_dislikes=("dislikes", "mean")
    )
    .reset_index()
)

# Show the first 10 rows of these time stats
display(interval_stats.head(10))

,publish_interval,video_count,avg_likes,avg_dislikes
0,2006-07-23 08:20:00+00:00,1,459.000000,152.000000
1,2007-03-05 16:20:00+00:00,9,336.666667,2.000000
2,2007-06-25 06:50:00+00:00,12,579.833333,11.500000
3,2007-12-03 20:50:00+00:00,16,187.937500,15.687500
4,2008-01-07 21:20:00+00:00,10,99.900000,2.000000
5,2008-01-13 01:30:00+00:00,2,1417.000000,49.500000
6,2008-02-12 20:20:00+00:00,3,1985.666667,124.666667
7,2008-04-05 18:20:00+00:00,4,46.000000,6.000000
8,2008-06-17 00:00:00+00:00,4,469.000000,4.000000
9,2008-08-07 12:10:00+00:00,3,78.333333,1.000000


## 8. Number of Videos for Each Tag

In [44]:
# Break the 'tags' column apart wherever there is a '|' symbol
consolidated_df["isolated_tags"] = consolidated_df["tags"].str.split("|")

# Take those lists of tags and give each individual tag its own separate row
tags_video_df = consolidated_df.explode("isolated_tags")

# Count how many videos use each specific tag
# Then, sort the list so the most popular tags are at the top
videos_per_tag = (
    tags_video_df
    .groupby("isolated_tags")
    .size()
    .reset_index(name="video_count")
    .sort_values("video_count", ascending=False)
)

# Show the top 10 most used tags in the whole dataset
display(videos_per_tag.head(10))

,isolated_tags,video_count
832565,[none],35518
336367,"""funny""",14834
277331,"""comedy""",11900
12345,"""2018""",10567
443877,"""news""",5653
436001,"""music""",5544
560941,"""video""",5338
11561,"""2017""",5334
363385,"""humor""",4992
535015,"""television""",4099


## 9. Tags with the Largest Number of Videos

In [61]:
# Sort the tags so the ones used in the most videos show up first
# This puts the biggest numbers at the very top of the list
top_tags = videos_per_tag.sort_values(
    by="video_count", ascending=False
)

# Show just the top 10 most popular tags
display(top_tags.head(10))

,isolated_tags,video_count
832565,[none],35518
336367,"""funny""",14834
277331,"""comedy""",11900
12345,"""2018""",10567
443877,"""news""",5653
436001,"""music""",5544
560941,"""video""",5338
11561,"""2017""",5334
363385,"""humor""",4992
535015,"""television""",4099


## 10. Average Like Ratio for Each (Tag, Country) Pair

In [62]:
# Chop the 'tags' column into a list of words wherever there is a '|' symbol
consolidated_df["isolated_tags"] = consolidated_df["tags"].str.split("|")

# Give each tag its own row while making sure we don't lose the country info
tags_country_df = consolidated_df.explode("isolated_tags")

# Count how many times each tag appears in each specific country
# This lets us see what's trending in different parts of the world
videos_per_tag_country = (
    tags_country_df
    .groupby(["isolated_tags", "country"])
    .size()
    .reset_index(name="video_count")
)

# Sort everything so the most popular combinations are at the top
# Then show the top 20 results
display(
    videos_per_tag_country
    .sort_values("video_count", ascending=False)
    .head(20)
)

,isolated_tags,country,video_count
1072469,[none],MX,7351
1072468,[none],KR,6739
1072464,[none],FR,5074
1072470,[none],RU,3641
455337,"""funny""",US,3576
455328,"""funny""",CA,3361
1072467,[none],JP,2958
1072463,[none],DE,2868
374369,"""comedy""",US,2851
455331,"""funny""",GB,2570


## 11. Most Viewed Video for Each (Trending Date, Country)

In [63]:
# For every single day and every country, find the row for the video that got the most views
idx = consolidated_df.groupby(
    ["trending_date", "country"]
)["views"].idxmax()

# Use those row numbers to grab only those "winner" videos for each day and country
top_video_per_day_country = consolidated_df.loc[idx].reset_index(drop=True)

# Show the first 20 videos that made it to the top of the list
display(top_video_per_day_country.head(20))

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_interval,isolated_tags
0,6ZfuNTqbHE8,17.01.12,Marvel Studios' Avengers: Infinity War Officia...,Marvel Entertainment,24,2017-11-29 13:26:24+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",56367282,2157741,34078,303178,https://i.ytimg.com/vi/6ZfuNTqbHE8/default.jpg,False,False,False,There was an ideaâ¦ Avengers: Infinity War. I...,CA,63.317712,2017-11-29 13:20:00+00:00,"[marvel, ""comics"", ""comic books"", ""nerdy"", ""ge..."
1,6ZfuNTqbHE8,17.01.12,Marvel Studios' Avengers: Infinity War Officia...,Marvel Entertainment,24,2017-11-29 13:26:24+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",56367282,2157737,34077,303178,https://i.ytimg.com/vi/6ZfuNTqbHE8/default.jpg,False,False,False,There was an ideaâ¦ Avengers: Infinity War. I...,DE,63.319453,2017-11-29 13:20:00+00:00,"[marvel, ""comics"", ""comic books"", ""nerdy"", ""ge..."
2,3VbHg5fqBYw,17.01.12,Avengers: Infinity War Trailer Tease,Marvel Entertainment,24,2017-11-28 17:09:22+00:00,"marvel""|""comics""|""comic books""|""nerdy""|""geeky""...",7281189,180808,19955,21244,https://i.ytimg.com/vi/3VbHg5fqBYw/default.jpg,False,False,False,Thank you to the best fans in the universe! Ma...,FR,9.060787,2017-11-28 17:00:00+00:00,"[marvel"", ""comics"", ""comic books"", ""nerdy"", ""g..."
3,TyHvyGVs42U,17.01.12,"Luis Fonsi, Demi Lovato - Ãchame La Culpa",LuisFonsiVEVO,10,2017-11-17 05:00:01+00:00,"Luis|""Fonsi""|""Demi""|""Lovato""|""Ãchame""|""La""|""C...",143408235,2686169,137938,144217,https://i.ytimg.com/vi/TyHvyGVs42U/default.jpg,False,False,False,âÃchame La Culpaâ disponible ya en todas ...,GB,19.473742,2017-11-17 05:00:00+00:00,"[Luis, ""Fonsi"", ""Demi"", ""Lovato"", ""Ãchame"", ""..."
4,6ZfuNTqbHE8,17.01.12,Marvel Studios' Avengers: Infinity War Officia...,Marvel Entertainment,24,2017-11-29 13:26:24+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",56367282,2157733,34077,303178,https://i.ytimg.com/vi/6ZfuNTqbHE8/default.jpg,False,False,False,There was an ideaâ¦ Avengers: Infinity War. I...,IN,63.319336,2017-11-29 13:20:00+00:00,"[marvel, ""comics"", ""comic books"", ""nerdy"", ""ge..."
5,6ZfuNTqbHE8,17.01.12,Marvel Studios' Avengers: Infinity War Officia...,Marvel Entertainment,24,2017-11-29 13:26:24+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",56370607,2157745,34078,303178,https://i.ytimg.com/vi/6ZfuNTqbHE8/default.jpg,False,False,False,There was an ideaâ¦ Avengers: Infinity War. I...,KR,63.31783,2017-11-29 13:20:00+00:00,"[marvel, ""comics"", ""comic books"", ""nerdy"", ""ge..."
6,6ZfuNTqbHE8,17.01.12,Marvel Studios' Avengers: Infinity War Officia...,Marvel Entertainment,24,2017-11-29 13:26:24+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",56370607,2157755,34079,303170,https://i.ytimg.com/vi/6ZfuNTqbHE8/default.jpg,False,False,False,There was an ideaâ¦ Avengers: Infinity War. I...,MX,63.316265,2017-11-29 13:20:00+00:00,"[marvel, ""comics"", ""comic books"", ""nerdy"", ""ge..."
7,3VbHg5fqBYw,17.01.12,Avengers: Infinity War Trailer Tease,Marvel Entertainment,24,2017-11-28 17:09:22+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",7281189,180808,19955,21244,https://i.ytimg.com/vi/3VbHg5fqBYw/default.jpg,False,False,False,Thank you to the best fans in the universe! Ma...,RU,9.060787,2017-11-28 17:00:00+00:00,"[marvel, ""comics"", ""comic books"", ""nerdy"", ""ge..."
8,6ZfuNTqbHE8,17.01.12,Marvel Studios' Avengers: Infinity War Officia...,Marvel Entertainment,24,2017-11-29 13:26:24+00:00,"marvel|""comics""|""comic books""|""nerdy""|""geeky""|...",56367282,2157727,34077,303178,https://i.ytimg.com/vi/6ZfuNTqbHE8/default.jpg,False,False,False,There was an ideaâ¦ Avengers: Infinity War. I...,US,63.31916,2017-11-29 13:20:00+00:00,"[marvel, ""comics""

## 12. Split Trending Date into Year, Month, and Day

In [64]:
# Change the 'trending_date' column into a format the computer actually recognizes as a date
# We tell it the date is written as Year.Day.Month
# If a date is broken or looks wrong, just leave it blank (NaT)
consolidated_df["trending_date"] = pd.to_datetime(
    consolidated_df["trending_date"], format="%y.%d.%m", errors="coerce"
)

# Pull just the year out of the date and put it in its own column
consolidated_df["year"] = consolidated_df["trending_date"].dt.year

# Pull just the month out and put it in its own column
consolidated_df["month"] = consolidated_df["trending_date"].dt.month

# Pull just the day out and put it in its own column
consolidated_df["day"] = consolidated_df["trending_date"].dt.day

# Show the original date alongside the new year, month, and day columns
display(consolidated_df[["trending_date", "year", "month", "day"]].head(20))

,trending_date,year,month,day
0,2017-11-14,2017,11,14
1,2017-11-14,2017,11,14
2,2017-11-14,2017,11,14
3,2017-11-14,2017,11,14
4,2017-11-14,2017,11,14
5,2017-11-14,2017,11,14
6,2017-11-14,2017,11,14
7,2017-11-14,2017,11,14
8,2017-11-14,2017,11,14
9,2017-11-14,2017,11,14


## 13. Most Viewed Video for Each (Month, Country)

In [65]:
# In every country, look at each month and find the row number for the video with the most views
idx = consolidated_df.groupby(
    ["month", "country"]
)["views"].idxmax()

# Use those row numbers to pick out the "monthly winner" for each country
top_video_per_month_country = consolidated_df.loc[idx].reset_index(drop=True)

# Show the first 20 rows of these top-performing videos
display(top_video_per_month_country.head(20))

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,ratings_disabled,video_error_or_removed,description,country,like_ratio,publish_interval,isolated_tags,year,month,day
0,LsoLEjrDogU,2018-01-09,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",43067983,1717177,61567,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,CA,27.891192,2018-01-04 04:40:00+00:00,"[Bruno Mars, ""Finesse"", ""Cardi B"", ""Finesse Re...",2018,1,9
1,LsoLEjrDogU,2018-01-08,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",37728802,1629946,56305,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,DE,28.948513,2018-01-04 04:40:00+00:00,"[Bruno Mars, ""Finesse"", ""Cardi B"", ""Finesse Re...",2018,1,8
2,LsoLEjrDogU,2018-01-08,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars""|""Finesse""|""Cardi B""|""Finesse Remix...",37728802,1629948,56305,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,FR,28.948548,2018-01-04 04:40:00+00:00,"[Bruno Mars"", ""Finesse"", ""Cardi B"", ""Finesse R...",2018,1,8
3,LsoLEjrDogU,2018-01-18,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",90598955,2248693,93089,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,GB,24.156377,2018-01-04 04:40:00+00:00,"[Bruno Mars, ""Finesse"", ""Cardi B"", ""Finesse Re...",2018,1,18
4,dfnCAmr569k,2018-01-18,"Taylor Swift - End Game ft. Ed Sheeran, Future",TaylorSwiftVEVO,10,2018-01-12 05:00:01+00:00,"Taylor|""Swift""|""End""|""Game""|""Big""|""Machine""|""Pop""",42019590,1804377,100033,...,False,False,Music video by Taylor Swift performing End Gam...,IN,18.037818,2018-01-12 05:00:00+00:00,"[Taylor, ""Swift"", ""End"", ""Game"", ""Big"", ""Machi...",2018,1,18
5,LsoLEjrDogU,2018-01-08,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",37728802,1629948,56304,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,KR,28.949062,2018-01-04 04:40:00+00:00,"[Bruno Mars, ""Finesse"", ""Cardi B"", ""Finesse Re...",2018,1,8
6,LsoLEjrDogU,2018-01-07,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",31680160,1510636,49497,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,MX,30.519749,2018-01-04 04:40:00+00:00,"[Bruno Mars, ""Finesse"", ""Cardi B"", ""Finesse Re...",2018,1,7
7,dfnCAmr569k,2018-01-14,"Taylor Swift - End Game ft. Ed Sheeran, Future",TaylorSwiftVEVO,10,2018-01-12 05:00:01+00:00,"Taylor|""Swift""|""End""|""Game""|""Big""|""Machine""|""Pop""",23198594,1404648,72534,...,False,False,Music video by Taylor Swift performing End Gam...,RU,19.365373,2018-01-12 05:00:00+00:00,"[Taylor, ""Swift"", ""End"", ""Game"", ""Big"", ""Machi...",2018,1,14
8,LsoLEjrDogU,2018-01-12,Bruno Mars - Finesse (Remix) [Feat. Cardi B] [...,Bruno Mars,10,2018-01-04 04:49:43+00:00,"Bruno Mars|""Finesse""|""Cardi B""|""Finesse Remix""...",57951412,1919583,73239,...,False,False,Finesse (Remix) Feat. Cardi B Available Now: h...,US,26.209847,2018-01-04 04:40:00+00:00,"[Bruno Mars, ""Finesse"", ""Cardi B"", ""Finesse Re...",2018,1,12
9,xpVfcZ0ZcFM,2018-02-24,Drake - Godâs Plan,DrakeVEVO,10,2018-02-17 05:00:01+00:00,"Drake new music|""Drake Gods Plan""|""Drake Godâ...",47362934,2469057,31843,...,False,False,Godâs Plan (Official Video)\n\nSong Availabl...,CA,77.538454,2018-02-17 05:00:00+00:00,"[Drake new music, ""Drake Gods Plan"", ""Drake Go...",2018,2,24


## 14. Read All JSON Files with Video Categories

In [66]:
# Create a blank list to hold our category info
categories = []
# Look through all folders to find every file that ends in .json
json_files = glob.glob("data/**/*.json", recursive=True)

for file in json_files:
    # Figure out the country by looking at the first part of the filename
    country = os.path.basename(file).split("_")[0].strip().upper()
    
    # Open each JSON file and read the data
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        # Go through the list of items inside the file
        for item in data.get("items", []):
            # Save the category ID, the name, and if it's usable for that country
            categories.append({
                "category_id": str(item.get("id")), 
                "category_name": item["snippet"]["title"],
                "assignable": bool(item.get("snippet", {}).get("assignable", True)),
                "country": country
            })

# Put everything into a table and remove any accidental duplicates
categories_df = pd.DataFrame(categories).drop_duplicates(subset=["country", "category_id"])

print("Category mapping extracted successfully.")
display(categories_df.head())

Category mapping extracted successfully.


,category_id,category_name,assignable,country
0,1,Film & Animation,True,CA
1,2,Autos & Vehicles,True,CA
2,10,Music,True,CA
3,15,Pets & Animals,True,CA
4,17,Sports,True,CA


## 15. Unassignable Categories per Country

In [67]:
# 1. Clean up the country and category IDs so they match perfectly
# We make sure everything is text, remove extra spaces, and make country codes uppercase
consolidated_df["country"] = consolidated_df["country"].astype(str).str.strip().str.upper()
consolidated_df["category_id"] = consolidated_df["category_id"].astype(str).str.strip()

# 2. Combine the video data with our "Translator" table from the last step
# This adds the actual category names (like 'Music') to our main list
master_df = pd.merge(
    consolidated_df, 
    categories_df, 
    on=["country", "category_id"], 
    how="left"
)

# 3. Look for two specific problems:
# Problem A: The category exists, but YouTube says it can't be used ('assignable' is False)
not_assignable = master_df[master_df["assignable"] == False]

# Problem B: The category ID isn't in our translator at all (it's just blank/NaN)
missing_mapping = master_df[master_df["assignable"].isna()]

# 4. Make a final report showing these issues for each country
all_countries = sorted(consolidated_df["country"].unique())
summary = pd.DataFrame(index=all_countries)

# Count how many of each problem we found per country
summary["not_assignable_count"] = not_assignable.groupby("country").size()
summary["missing_category_mapping"] = missing_mapping.groupby("country").size()

# Fill any empty spots with 0 and show the final table
summary = summary.fillna(0).astype(int)
print("\nPer-country summary (not assignable / missing mapping)")
display(summary)


Per-country summary (not assignable / missing mapping)


,not_assignable_count,missing_category_mapping
CA,130,69
DE,102,228
FR,110,85
GB,20,90
IN,220,41
JP,0,18
KR,167,280
MX,3,149
RU,183,1301
US,57,0
